In [1]:
import pandas as pd

In [2]:
students_url = "https://drive.google.com/uc?export=download&id=1P6eZPCg1FwpXnbq2Yt52RTdvDLk4PSD9"
students = pd.read_csv(students_url, compression="gzip")

pd.set_option("display.max_columns", None)

students.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,weekly_work_hours,course_type,study_mode,semester,international_student,admission_grade,previous_qualification_grade,units_enrolled,units_passed,units_failed,gpa_semester_1,gpa_semester_2,attendance_rate,assignments_submitted_rate,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,24,STEM,Online,7,0,164.6,165.1,11,11,0,3.44,2.55,77.3,75.0,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,0,Arts & Humanities,Full-time,6,1,175.2,164.1,14,11,3,2.56,2.53,52.1,80.1,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,16,Arts & Humanities,Full-time,3,0,126.0,179.3,17,14,3,3.43,2.73,100.0,67.8,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,0,Business,Full-time,1,0,75.5,161.9,13,11,2,2.51,2.10,83.9,100.0,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,0,Health Sciences,Part-time,2,0,89.0,101.8,8,7,1,3.93,2.88,61.6,100.0,4,7.4,1,0,7,0,10,24,5.8,0


In [3]:
students.shape

(100000, 34)

## Exercise 1

Extract all students that study Online, have an attendance_rate smaller than 50,
and have more than 5 missed_deadlines. Store them in a dataframe called at_risk.

In [4]:
study_mask = (students["study_mode"] == "Online") & (students["attendance_rate"] < 50) & (students["missed_deadlines"] > 5)
at_risk = students.loc[study_mask]
at_risk[["person_id", "study_mode", "attendance_rate", "missed_deadlines"]].head()

,person_id,study_mode,attendance_rate,missed_deadlines
478,479,Online,49.7,8
581,582,Online,46.7,11
642,643,Online,46.1,9
1216,1217,Online,32.6,7
1316,1317,Online,35.5,17


## Exercise 2
Then compute, for each course_type, how many of those students there are and what fraction of them has dropped out. 

In [5]:
course_type_counts = at_risk["course_type"].value_counts()
course_type_counts

course_type
Business                 47
STEM                     47
Arts & Humanities        40
Health Sciences          34
Law & Social Sciences    29
Name: count, dtype: int64

In [6]:
course_type_dropouts_counts = at_risk.loc[at_risk["dropout"] == 1, "course_type"].value_counts()
course_type_dropouts_counts

course_type
Business                 36
STEM                     36
Arts & Humanities        30
Law & Social Sciences    26
Health Sciences          22
Name: count, dtype: int64

In [7]:
course_dropout_rate = course_type_dropouts_counts / course_type_counts
course_dropout_rate

course_type
Arts & Humanities        0.750000
Business                 0.765957
Health Sciences          0.647059
Law & Social Sciences    0.896552
STEM                     0.765957
Name: count, dtype: float64

#### Store the result in a dataframe called at_risk_by_course, with columns students and dropout_rate, sorted by decreasing dropout_rate.

In [8]:
at_risk_by_course = pd.DataFrame({
    "students": course_type_counts,
    "dropout_rate": course_dropout_rate,
})

at_risk_by_course = at_risk_by_course.sort_values(by = "dropout_rate", ascending=False)
at_risk_by_course

,students,dropout_rate
course_type,,
Law & Social Sciences,29,0.896552
Business,47,0.765957
STEM,47,0.765957
Arts & Humanities,40,0.750000
Health Sciences,34,0.647059


## Exercise 3

Add to the dataframe the following columns:

- `pass_rate`: the ratio between `units_passed` and `units_enrolled`.
- `gpa_change`: the difference between `gpa_semester_2` and `gpa_semester_1`.
- `gpa_trend`: "improving" if `gpa_change` is larger than 0.5, "declining" if it is smaller than -0.5, and "stable" otherwise.

In [9]:
students["pass_rate"] = students["units_passed"] / students["units_enrolled"]
students[["units_passed", "units_enrolled", "pass_rate"]].head()

,units_passed,units_enrolled,pass_rate
0,11,11,1.000000
1,11,14,0.785714
2,14,17,0.823529
3,11,13,0.846154
4,7,8,0.875000


In [10]:
students["gpa_change"] = students["gpa_semester_2"] - students["gpa_semester_1"]

students["gpa_change"]

0       -0.89
1       -0.03
2       -0.70
3       -0.41
4       -1.05
         ... 
99995   -0.98
99996    0.21
99997   -0.57
99998   -0.99
99999   -0.78
Name: gpa_change, Length: 100000, dtype: float64

In [11]:
students["gpa_trend"] = "stable"

students.loc[students["gpa_change"] > 0.5, "gpa_trend"] = "improving"

students.loc[students["gpa_change"] < - 0.5, "gpa_trend"] = "declining"

students[["gpa_change", "gpa_trend"]]

,gpa_change,gpa_trend
0,-0.89,declining
1,-0.03,stable
2,-0.70,declining
3,-0.41,stable
4,-1.05,declining
...,...,...
99995,-0.98,declining
99996,0.21,stable
99997,-0.57,declining
99998,-0.99,declining


## Exercise 4
Verify that, for every student, units_passed + units_failed equals units_enrolled; <br>
if not, report the rows violating this condition.

In [12]:
enrolled_mask = students["units_passed"] + students["units_failed"] != students["units_enrolled"]
violations = students.loc[enrolled_mask]

if len(violations) == 0:
    print("All students satisfy the condition.")
else:
    print("These students violate the condition:")
    print(violations)

All students satisfy the condition.


## Exercise 5
Compute the dropout rate for each value of gpa_trend, and the person_id of the 10 students with the largest GPA drop.

In [13]:
gpa_trend_counts = students["gpa_trend"].value_counts()
gpa_trend_counts

gpa_trend
declining    57536
stable       41763
improving      701
Name: count, dtype: int64

In [14]:
gpa_trend_dropout_counts = students.loc[students["dropout"] == 1, "gpa_trend"].value_counts()
gpa_trend_dropout_counts

gpa_trend
stable       16339
declining    15252
improving      413
Name: count, dtype: int64

In [15]:
gpa_trend_dropout_rate = gpa_trend_dropout_counts / gpa_trend_counts
gpa_trend_dropout_rate

gpa_trend
declining    0.265086
improving    0.589158
stable       0.391231
Name: count, dtype: float64

In [16]:
largest_drop = students.sort_values("gpa_change", ascending = True)
largest_drop[["person_id", "gpa_change"]].head(10)

,person_id,gpa_change
23225,23226,-2.49
96224,96225,-2.40
88489,88490,-2.37
51824,51825,-2.35
58935,58936,-2.33
46714,46715,-2.32
43035,43036,-2.30
72663,72664,-2.27
76705,76706,-2.23
23427,23428,-2.23


## Exercise 6 
Partition the students into the following groups, according to weekly_work_hours: <br>
0, 1-10, 11-20, 21-30, more than 30 (all intervals include both extremes).

In [17]:
students["weekly_work_group"] = "0"

second_mask = (students["weekly_work_hours"] >= 1) & (students["weekly_work_hours"] <= 10)
students.loc[second_mask, "weekly_work_group"] = "1-10"

third_mask = (students["weekly_work_hours"] >= 11) & (students["weekly_work_hours"] <= 20)
students.loc[third_mask, "weekly_work_group"] = "11-20"

forth_mask = (students["weekly_work_hours"] >= 21) & (students["weekly_work_hours"] <= 30)
students.loc[forth_mask, "weekly_work_group"] = "21-30"

fifth_mask = students["weekly_work_hours"] > 30
students.loc[fifth_mask, "weekly_work_group"] = "more than 30"

students[["weekly_work_hours", "weekly_work_group"]].head(10)

,weekly_work_hours,weekly_work_group
0,24,21-30
1,0,0
2,16,11-20
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


## Exercise 7

Build a risk score by summing:

- 2 points if the tuition is not up to date.
- 1 point if `stress_level` is at least 8.
- 1 point if `sleep_hours` is smaller than 5.
- 1 point if the student has a part-time job with more than 20 weekly work hours.
- 2 points if `attendance_rate` is smaller than 60.
- 1 point if `missed_deadlines` is larger than 3.

For each value of the risk score, compute the number of students and the dropout rate.

In [18]:
students["risk_score"] = 0

students.loc[students["tuition_up_to_date"] == 0, "risk_score"] += 2

students.loc[students["stress_level"] >= 8, "risk_score"] += 1

students.loc[students["sleep_hours"] < 5, "risk_score"] += 1

work_mask = (students["part_time_job"] == 1) & (students["weekly_work_hours"] > 20)
students.loc[work_mask, "risk_score"] += 1

students.loc[students["attendance_rate"] < 60, "risk_score"] += 2

students.loc[students["missed_deadlines"] > 3, "risk_score"] += 1

students[["risk_score"]].head()

,risk_score
0,4
1,2
2,1
3,0
4,0


In [19]:
risk_score_counts = students["risk_score"].value_counts()
risk_score_counts

risk_score
0    26955
1    24115
2    21984
3    14557
4     7613
5     3320
6     1149
7      267
8       40
Name: count, dtype: int64

In [20]:
risk_score_dropout_counts = students.loc[students["dropout"] == 1, "risk_score"].value_counts()
risk_score_dropout_counts

risk_score
2    7413
3    6590
1    6079
4    4396
0    4120
5    2236
6     913
7     221
8      36
Name: count, dtype: int64

In [21]:
risk_score_dropout_rate = risk_score_dropout_counts / risk_score_counts

risk_score_summary = pd.DataFrame({
    "students": risk_score_counts,
    "dropout_rate": risk_score_dropout_rate
})

risk_score_summary

,students,dropout_rate
risk_score,,
0,26955,0.152847
1,24115,0.252084
2,21984,0.337200
3,14557,0.452703
4,7613,0.577433
5,3320,0.673494
6,1149,0.794604
7,267,0.827715
8,40,0.900000


## Exercise 8
Read the GDP per capita dataset published by Our World in Data directly from the URL <br>
and store it in a dataframe called gdp. The column ny_gdp_pcap_pp_kd contains the GDP per capita.

In [22]:
gdp = pd.read_csv(
    "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true")
gdp.head()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia


#### Clean the dataset, removing the rows that do not correspond to a single country (e.g. World, High-income countries).

In [23]:
entity_names = pd.Series(gdp["entity"].unique())
print(entity_names.to_string(index=False))

                                       Afghanistan
                                           Albania
                                           Algeria
                                           Andorra
                                            Angola
                               Antigua and Barbuda
                                         Argentina
                                           Armenia
                                             Aruba
                                         Australia
                                           Austria
                                        Azerbaijan
                                           Bahamas
                                           Bahrain
                                        Bangladesh
                                          Barbados
                                           Belarus
                                           Belgium
                                            Belize
                               

In [24]:
gdp_mask = gdp["entity"].isin(["World", "High-income countries", "East Asia and Pacific (WB)", "Upper-middle-income countries", "Sub-Saharan Africa (WB)",
                   "North America (WB)", "Middle East, North Africa, Afghanistan and Pakistan (WB)",
                   "Lower-middle-income countries", "Low-income countries", "Latin America and Caribbean (WB)",
                   "European Union (27)", "Europe and Central Asia (WB)", "South Asia (WB)"])
gdp = gdp.loc[~gdp_mask]
gdp

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
...,...,...,...,...,...
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa


#### For each country keep only the row with the most recent year.

In [25]:
gdp = gdp.sort_values(by="year", ascending=False)
gdp = gdp.drop_duplicates(subset = "entity", keep= "first")
gdp.head()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
7444,Zimbabwe,ZWE,2025,5529.2330,Africa
4855,Nicaragua,NIC,2025,7933.4890,North America
1336,Chile,CHL,2025,30863.4220,South America
4639,Myanmar,MMR,2025,5137.1006,Asia
1480,Congo,COG,2025,6194.0900,Africa


## Exercise 9 
Join the GPD data with the students data. Compute the dropout rate for each continent.

In [26]:
unmatched = ~students["nationality"].isin(gdp["entity"]) 
students.loc[unmatched, "nationality"].unique()

<ArrowStringArray>
['USA', 'UK']
Length: 2, dtype: str

In [27]:
gdp["entity"] = gdp["entity"].replace({"United States": "USA"})
gdp["entity"] = gdp["entity"].replace({"United Kingdom": "UK"})

unmatched = ~students["nationality"].isin(gdp["entity"])
students["nationality"].loc[unmatched].unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

In [28]:
joined_df = students.merge(gdp,
               left_on = "nationality",
               right_on = "entity",
               how = "left")
joined_df[["person_id", "nationality", "entity", "owid_region"]].head()

,person_id,nationality,entity,owid_region
0,1,Brazil,Brazil,South America
1,2,USA,USA,North America
2,3,USA,USA,North America
3,4,USA,USA,North America
4,5,Canada,Canada,North America


In [29]:
continent_count = joined_df["owid_region"].value_counts()
continent_count

owid_region
North America    38095
Asia             36035
Europe           10122
Africa            7906
South America     6873
Oceania            969
Name: count, dtype: int64

In [30]:
continent_dropout_counts = joined_df.loc[joined_df["dropout"] == 1, "owid_region"].value_counts()
continent_dropout_counts

owid_region
North America    12102
Asia             11636
Europe            3267
Africa            2504
South America     2183
Oceania            312
Name: count, dtype: int64

In [31]:
dropout_continent = continent_dropout_counts / continent_count
dropout_continent

dropout_continent_summary = pd.DataFrame({
    "students": continent_count,
    "dropout_rate": dropout_continent
})

dropout_continent_summary

,students,dropout_rate
owid_region,,
North America,38095,0.317679
Asia,36035,0.322908
Europe,10122,0.322762
Africa,7906,0.316721
South America,6873,0.317620
Oceania,969,0.321981


## Use of Generative AI

I used ChatGPT as a learning and review aid while developing this project. I wrote initial solutions and used it to check their correctness, understand errors, and suggest more compact alternatives to some code.
I also used it to explain pandas syntax and validate my interpretations of the code. During revision, I used it to help compare outputs and check whether changes preserved the results. Finally, I used it to check for mistakes.

I reviewed the suggestions before incorporating them and used the explanations to develop my understanding of the methods applied.